<a href="https://colab.research.google.com/github/dinukadilshan03/RoadLens/blob/feature%2Fc1-base-model-training/ml/c1/notebooks/train_base_model_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base Model Training Pipeline (B0 Baseline)

This notebook trains the **Base Model (B0 Baseline)** for **Component 1 (C1: Road Damage Detection & Temporal Consolidation)** using a YOLOv8m detector.

### Pipeline Configuration
- **Architecture:** `YOLOv8m` (25.9M parameters)
- **Dataset Config:** `ml/c1/configs/base_model_b0.yaml` (4 core distress classes: `D00`, `D10`, `D20`, `D40`)
- **Hyperparameters:** `ml/c1/configs/hyp_base.yaml` (AdamW, cosine learning rate scheduler, road distress augmentations)
- **Resolution:** `800x800` (preserves fine sub-surface crack features)
- **Primary Checkpoint:** `base_model_b0.pt`

### 1. Hardware & Compute Environment Verification

Queries the NVIDIA system management interface (`nvidia-smi`) to inspect available GPU hardware, driver version, CUDA capability, and total video memory (VRAM). Ensures the PyTorch CUDA runtime detects an active accelerator before initializing model weights.

In [ ]:
!nvidia-smi

### 2. Persistent Storage Mount (Google Drive)

Mounts Google Drive to `/content/drive` to establish persistent storage. Because Colab execution environments are ephemeral and destroyed upon disconnection, checkpoints (`base_model_b0.pt`), loss curves, and evaluation metrics are copied to Drive to ensure zero data loss.

In [ ]:
import os
from google.colab import drive

drive_mount_point = '/content/drive'
if not os.path.exists(drive_mount_point):
    drive.mount(drive_mount_point)
    print("Google Drive mounted successfully.")
else:
    print(f"Google Drive already mounted at {drive_mount_point}.")

### 3. Repository Provisioning & Environment Setup

Clones or synchronizes the `feature/c1-base-model-training` branch from the RoadLens repository and configures the execution environment:
- **Idempotent Repository Management:** If `/content/RoadLens` already exists, it fetches and resets to the latest branch commit instead of failing with directory collision errors.
- **Dependency Installation:** Installs `ultralytics` and `pyyaml` without forcing upgrades on existing system libraries (`pandas`, `google-colab`), avoiding environment dependency conflicts.

In [ ]:
import os
from pathlib import Path

repo_dir = Path('/content/RoadLens')
if not repo_dir.exists():
    !git clone -b feature/c1-base-model-training https://github.com/dinukadilshan03/RoadLens.git /content/RoadLens
else:
    print("Repository directory exists. Synchronizing with latest origin/feature/c1-base-model-training...")
    !git -C /content/RoadLens fetch origin feature/c1-base-model-training
    !git -C /content/RoadLens checkout feature/c1-base-model-training
    !git -C /content/RoadLens reset --hard origin/feature/c1-base-model-training

# Set active working directory to repository root
%cd /content/RoadLens

# Install YOLO framework and YAML parser while preserving Colab system dependencies
!pip install --quiet "ultralytics>=8.3.0" pyyaml

### 4. Dataset Extraction & Verification

Prepares the 4-class road distress dataset for model training:
- **Exact Archive Path:** Loads `rdd_cleaned.zip` directly from Google Drive at `/content/drive/MyDrive/data/rdd_cleaned.zip`.
- **Extraction Guard:** Inspects `ml/c1/data/rdd_cleaned/train/images` before unzipping. If already extracted, redundant unzipping is bypassed.
- **Integrity Verification:** Asserts that `train`, `val`, and `test` splits contain valid image files and logs split counts.

In [ ]:
import os
import shutil
from pathlib import Path

target_data_dir = Path('/content/RoadLens/ml/c1/data/rdd_cleaned')
target_train_dir = target_data_dir / 'train' / 'images'

# Check if dataset is already extracted
if target_train_dir.exists() and any(target_train_dir.iterdir()):
    train_count = len(list(target_train_dir.glob('*.jpg')))
    print(f"Dataset already extracted at {target_data_dir} ({train_count} training images found). Skipping extraction.")
else:
    # Exact path to dataset archive in Google Drive
    zip_path = Path('/content/drive/MyDrive/data/rdd_cleaned.zip')
    
    if not zip_path.is_file():
        raise FileNotFoundError(
            f"Dataset archive not found at exact path: {zip_path}. "
            "Please ensure 'rdd_cleaned.zip' exists inside your Google Drive 'data' folder."
        )

    print(f"Dataset archive detected: {zip_path}")
    print("Extracting dataset to /content/RoadLens/ml/c1/data/ (this may take ~1 minute)...")
    !unzip -q "{zip_path}" -d /content/RoadLens/ml/c1/data/
    print("Extraction complete.")

# Validate dataset split integrity
train_imgs = len(list((target_data_dir / 'train' / 'images').glob('*.*')))
val_imgs = len(list((target_data_dir / 'val' / 'images').glob('*.*')))
test_imgs = len(list((target_data_dir / 'test' / 'images').glob('*.*')))

print("\nDataset Split Verification Summary:")
print(f"  - Train: {train_imgs} images")
print(f"  - Val:   {val_imgs} images")
print(f"  - Test:  {test_imgs} images")
assert train_imgs > 0, "Train split is empty."
assert val_imgs > 0, "Val split is empty."


### 5. Base Model (B0) Training Execution

Executes `ml/c1/scripts/base_model_training.py` with the following configuration:
- `--data ml/c1/configs/base_model_b0.yaml`: Dataset configuration mapping class indices `0: Longitudinal_Crack_D00`, `1: Transverse_Crack_D10`, `2: Alligator_Crack_D20`, `3: Pothole_D40`.
- `--hyp ml/c1/configs/hyp_base.yaml`: Hyperparameter definitions including AdamW optimizer, cosine learning rate scheduler, box gain `7.5`, and pavement-aware augmentations.
- `--model yolov8m.pt`: YOLOv8 medium architecture pretrained on COCO.
- `--epochs 50`: Total epoch schedule with early stopping patience of 15 epochs.
- `--imgsz 800`: 800x800 input resolution to maintain crack feature map fidelity.
- `--batch 16`: Batch size balanced for VRAM headroom and batch-norm stability.
- `--device 0`: Targets primary CUDA accelerator.

In [ ]:
!python ml/c1/scripts/base_model_training.py \
    --data ml/c1/configs/base_model_b0.yaml \
    --hyp ml/c1/configs/hyp_base.yaml \
    --model yolov8m.pt \
    --name base_model_b0 \
    --epochs 50 \
    --imgsz 800 \
    --batch 16 \
    --device 0

### 6. Checkpoint Archival & Metrics Export

Manages post-training persistence to Google Drive:
- Copies the best model weights (`base_model_b0.pt`) to `/content/drive/MyDrive/RoadLens_C1_Weights/`.
- Archives training diagnostics (`ml/c1/runs/train/base_model_b0/`) including precision-recall curves (`PR_curve.png`), F1 curves (`F1_curve.png`), confusion matrices, and `results.csv` into a compressed `.tar.gz` archive.

In [ ]:
import os
import shutil
from pathlib import Path

drive_weights_dir = Path("/content/drive/MyDrive/RoadLens_C1_Weights")
drive_weights_dir.mkdir(parents=True, exist_ok=True)

# Check for primary checkpoint
best_weights = Path("ml/c1/weights/base_model_b0.pt")
if not best_weights.exists():
    # Fallback to run output directory if not yet copied locally
    best_weights = Path("ml/c1/runs/train/base_model_b0/weights/best.pt")

if best_weights.is_file():
    dest_weights = drive_weights_dir / "base_model_b0.pt"
    shutil.copy2(best_weights, dest_weights)
    print(f"Best model checkpoint saved to Drive: {dest_weights}")
else:
    print(f"Warning: Checkpoint not found at {best_weights}. Verify training status.")

# Archive complete training run plots and logs
run_train_dir = Path("ml/c1/runs/train")
if run_train_dir.exists():
    archive_dest = drive_weights_dir / "training_run_logs.tar.gz"
    !tar -czf "{archive_dest}" ml/c1/runs/train/
    print(f"Training logs and curves backed up to Drive: {archive_dest}")